# 04 — Fit practical and objective-matched controls

## Question

Does clean-target latent pretraining add value beyond matched coordinate learning and practical temporal denoising?

## Inputs

The information-stage receipt, the fixed train/development groups and an explicit run scope. CPU fixtures use small models and two-update phase checks. Source CUDA execution requires the pinned HAIC runtime and authorized, measured all-stage compute scope.

Use an explicit `STV2_CONFIG` JSON file and a unique run ID. The setup resolves relative artifact paths from the repository root. See the [execution guide](README.md), [development protocol](../../docs/studies/synthetic-training-v2/protocol.md) and [literature ledger](../../docs/studies/synthetic-training-v2/literature.md).

In [ ]:
import os
import sys
from pathlib import Path
from IPython.display import Image, display

if not os.environ.get("STV2_CONFIG"):
    raise RuntimeError("Set STV2_CONFIG to an explicit study JSON configuration before execution.")
config_path = Path(os.environ["STV2_CONFIG"]).expanduser().resolve()
if not config_path.is_file():
    raise FileNotFoundError(f"Study configuration does not exist: {config_path}")
search_root = Path(os.environ.get("GAVD6_ROOT", Path.cwd())).expanduser().resolve()
PROJECT_ROOT = next((path for path in (search_root, *search_root.parents)
                     if (path / "src/gavd6_sjepa").is_dir() and (path / "pyproject.toml").is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Run inside the repository or set GAVD6_ROOT to its root.")
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))
os.chdir(PROJECT_ROOT)
os.environ["STV2_CONFIG"] = str(config_path)
from gavd6_sjepa.research_directions.synthetic_training_v2.config import RunConfig
from gavd6_sjepa.research_directions.synthetic_training_v2.workflow import run_stage

cfg = RunConfig.load(os.environ["STV2_CONFIG"])
display({"run_id": cfg.run_id, "mode": cfg.mode, "device": cfg.device,
         "artifact_root": str(cfg.root), "confirmation": "closed"})
if cfg.mode == "fixture":
    print("CPU software fixture: method ordering is not empirical evidence.")

## Computation

Fit the configured practical and initialized arms first, then coordinate reconstruction, ordinary JEPA, paired JEPA and shuffled-pair controls. Each representation receives its own matched readout. The training modules own masks, loss support, EMA, optimization, checkpoints and predictions.

`run_stage` implements the computation in the study modules. It checks prerequisite receipts and returns the saved result on an unchanged rerun; a changed configuration or code identity requires a new run ID.

In [ ]:
direct = run_stage(cfg, "direct", repo_root=PROJECT_ROOT)
display({"evidence_status": direct["evidence_status"],
         "resource_contrast": direct["resource_contrast"],
         "equal_compute_claim": direct["equal_compute_claim"]})
display([{key: row.get(key) for key in
          ("arm", "seed", "status", "optimizer_updates", "elapsed_seconds", "termination")}
         for row in direct["fits"]])

In [ ]:
jepa = run_stage(cfg, "jepa", repo_root=PROJECT_ROOT)
display({"evidence_status": jepa["evidence_status"],
         "resource_contrast": jepa["resource_contrast"],
         "equal_compute_claim": jepa["equal_compute_claim"]})
display([{key: row.get(key) for key in
          ("arm", "seed", "status", "optimizer_updates", "elapsed_seconds", "termination")}
         for row in jepa["fits"]])

## Outputs and checks

Inspect per-arm `fits/<arm>-<seed>/training.json`, checkpoints and saved development predictions. Check actual updates, teacher lag, skipped loss support, input dependence, cost and trainable capacity. Displayed fit status describes execution, not superiority. No notebook submits a GPU job.

Stage receipts under `receipts/` record elapsed time and hashes of produced artifacts. Inspect the saved files for full diagnostics; the display above is deliberately brief.

## Interpretation

The SmoothNet-style 2D MLP is a practical comparator, not an exact paper reproduction. The per-frame control shares whole-window input normalization, so it tests added coordinate history conditional on that transform; it is not an absolute no-history baseline. PoseBERT and MotionBERT already use corrupted or partial pose reconstruction; S-JEPA motivates latent prediction. The strongest competing explanation is clean-target supervision or ordinary denoising. Paired versus coordinate is the objective contrast; ordinary JEPA changes supervision source. Equal steps do not establish equal compute.

A completed fixture checks software behavior. Scientific gates use `pass`, `fail` or `insufficient_evidence`; fixture success cannot make a scientific gate pass.

## Next gate

Proceed to [05 — evaluation](05_evaluation.ipynb). A fixture cannot choose finalists; decisive source comparisons require prespecified repeated seeds, adequate motion support and independently calibrated margins.